# Piping Chains and the RunnablePassthrough Class

In [1]:
pip show langchain

Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\AayushiTrivedi\.conda\envs\langchain_env\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext dotenv
%dotenv

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [5]:
RunnablePassthrough().invoke([1, 2, 3])

[1, 2, 3]

In [6]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [7]:
chat_template_tools

ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools.\n'), additional_kwargs={})])

In [8]:
chat = ChatGroq(model_name = "llama-3.1-8b-instant", 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 100)

In [9]:
string_parser = StrOutputParser()

In [10]:
chain_tools = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()})
chain_strategy = chat_template_strategy | chat | string_parser

In [11]:
print(chain_tools.invoke({'job title':'data scientist'}))

{'tools': '1. Python\n2. R\n3. Jupyter Notebook\n4. Tableau\n5. SQL'}


In [12]:
print(chain_strategy.invoke({'tools':'''
1. Python
2. R Programming
3. SQL
4. Tableau
5. Hadoop
'''}))

**Mastering the Tools: A Comprehensive Learning Strategy**

To effectively learn and master the tools listed, follow this structured approach:

**Phase 1: Fundamentals (Weeks 1-4)**

1. **Python**:
	* Start with basic syntax and data types (e.g., variables, data structures, control structures).
	* Learn popular libraries like NumPy, pandas, and Matplotlib for data manipulation and visualization.
	* Practice with online platforms like LeetCode, HackerRank


In [14]:
chain_combined = chain_tools | chain_strategy

In [15]:
print(chain_combined.invoke({'job title':'data scientist'}))

**Mastering the Tools: A Comprehensive Learning Strategy**

To effectively learn and master the tools listed, follow this structured approach:

**Phase 1: Fundamentals (Weeks 1-4)**

1. **Python**:
	* Start with basic syntax, data types, and control structures.
	* Learn popular libraries like NumPy, pandas, and matplotlib.
	* Practice with exercises and projects on platforms like LeetCode, HackerRank, or CodeWars.
2. **R**


In [16]:
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)